# 💸💸💸 <span style="color: white; background-color: SteelBlue"><b> Extração do Realizado x Orçado </b></span></p>

🧩 <span style="color: LightSteelBlue"><b> 1- Configuração inicial e registro do processo </b></span></p>
- Importa bibliotecas (Selenium, PyAutoGUI, OpenPyXL, Logging etc.)
- Define pastas de entrada/saída (Downloads, Processados, Exportados)
- Configura ID de processo
- Registra etapas no arquivo corporativo PROCESSOS.xlsx
- Isso garante auditoria e rastreabilidade em cada execução

🌐 <span style="color: LightSteelBlue"><b> 2- Acesso automático ao NetSuite (Hyperion Planning) </b></span></p>
- Abre o navegador no endereço do módulo de Orçamento  
- Realiza login automático (digitando usuário/senha via PyAutoGUI)  
- Aguarda carregamento da interface HTML5 do Hyperion  
- Posiciona o cursor no menu correto
- Esse é o ponto de partida para os blocos de extração

🏢 <span style="color: LightSteelBlue"><b> 3- Extração automatizada — Centros de Custo da SEDE </b></span></p> 
- Seleciona Realizado ou Orçado  
- Define o ano (ex.: 2026)  
- Seleciona a unidade 1 - SEDE  
- Atualiza o dropdown de centro de custo  
- Exporta a planilha  
- Aguarda o download  
- Renomeia o arquivo exportado  
- Move o arquivo para Centros_Custo_Exportados
- O processo possui
    - Laço automatizado  
    - Mensagens de progresso  
    - Tratamento de erros  
    - Delays calibrados para estabilidade do Hyperion

🏖 <span style="color: LightSteelBlue"><b> 4- Extração automatizada — UL e UR (Unidades de Lazer e Unidades Regionais) </b></span></p>
- Uma segunda rotina — semelhante à da SEDE — executa a extração para
    - ULs  
    - URs
- Isso permite obter o conjunto completo de centros de custo operacionais e regionais
- Arquivos são armazenados em Centros_Custo_UL_Exportados
- Ambas as rotinas garantem padronização do fluxo, mesmo com listas diferentes de centros.

📁 <span style="color: LightSteelBlue"><b> 5- Organização dos arquivos exportados </b></span></p>
- O script valida se os arquivos baixados foram movidos para o repositório consolidado  
- Exibe prompts ao usuário (PyAutoGUI) para confirmar movimentações  
- Evita duplicidade e perda de arquivos
- Essa etapa garante saneamento da estrutura de diretórios

📈 <span style="color: LightSteelBlue"><b> 6- Atualização automática do Power BI — Real x Orçado </b></span></p>
- O script abre REAL X ORÇADO.pbix
- E aguarda o carregamento do Power BI Desktop  
- Clica no botão de Atualizar  
- Aguarda conclusão  
- Clica em Publicar  
- Substitui o relatório no workspace GESTÃO DE PESSOAS  
- Fecha o Power BI
- Esse bloco substitui completamente a necessidade de:
    - Abrir manualmente o pbix  
    - Atualizar as fontes  
    - Publicar no serviço  
    - Confirmar sobrescrita
- E garante que os dados mais recentes (Real e Orçado) estejam sempre no dashboard corporativo

🧾 <span style="color: LightSteelBlue"><b> 7- Registro e métricas finais </b></span></p>
- O script calcula
    - Tempo total de execução  
    - Status final do processo
- E imprime um resumo
    - Quantos centros foram processados  
    - Execução concluída com sucesso  
    - Tempo gasto

# Importando as Bibliotecas

In [ ]:
import datetime
import glob
import logging
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Caminhos de Pastas

In [ ]:
CONFIG = {
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'padrao': 'HsWebGrid',
}

# Registra Etapa do Processo

In [ ]:
id = 26

path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'

registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')

wb_p = load_workbook(path_registros_processos)

ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0

tempo_0 = [id, datetime.today(), 0]

ws_p.append(tempo_0)

wb_p.save(path_registros_processos)

# Acessando o NetSuite

In [ ]:
# Configuração
chrome_options = Options()
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-allow-origins=*")
chrome_options.add_argument("--start-maximized")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

load_dotenv(dotenv_path='env_path')

epmcloud_user = os.getenv("EPMCLOUD_USER")
epmcloud_password = os.getenv("EPMCLOUD_PASSWORD")
epmcloud_site = os.getenv("SITE_EPMCLOUD")

try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
        driver.get(epmcloud_site)
        wait = WebDriverWait(driver, 20)
        actions = ActionChains(driver)
        time.sleep(1)
        window = win32gui.GetForegroundWindow()
        win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
        
        print("⏳ Sincronizando NetSuite...")
        
        print("🏁 Acesso inicializado. Executando tarefa de login...")
        pyautogui.PAUSE = 1

        time.sleep(5) # Pequena pausa para garantir renderização visual para o PyAutoGUI
        
        pyautogui.write(epmcloud_user)
        pyautogui.press("tab")
        pyautogui.write(epmcloud_password)
        pyautogui.press("enter")
        
        print("⏳ Aguardando carregamento da página (20s)...")
        time.sleep(20) # Tempo para carregar a página de login do usuário

except Exception as e:
        print(f'❌ Erro: {str(e)}')

print('----------------------------------------------------------------------------------------------------')
print('   ✅ Acesso realizado com sucesso')
print('----------------------------------------------------------------------------------------------------')

# Acessando o Módulo Realizado - Centros de Custo da SEDE

In [ ]:
# Configuração do logging
logging.basicConfig(
    filename='automacao_centros_custo.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Lista completa de centros de custo
CENTROS_CUSTO_SEDE = [
    "SECRETARIA DE EVENTOS",
    "ACADEMIA DE LETRAS",
    "ASSESSORIA JURIDICA",
    "ASSESSORIA PRESIDENCIA",
    "SECRETARIA PRESIDENCIA",
    "MARKETING",
    "OUVIDORIA",
    "DEPARTAMENTO PESSOAL",
    "COORD. DE ASSOCIATIVISMO",
    "CAIXA GERAL",
    "CONTROLADORIA",
    "CENTRAL DE RELACIONAMENTO",
    "TRANSPORTES",
    "SUPRIMENTOS E LOGISTICA",
    "COORD. SOCIAL",
    "PROTOCOLO",
    "COORD. ASSISTENCIA SAUDE",
    "COORD. DE ESPORTES",
    "SECRETARIA DE TURISMO",
    "SECRETARIA DE PATRIMONIO",
    "COORD. DO MEIO AMBIENTE",
    "COORD. GESTAO ADM URS",
    "COORD. DAS ULS",
    "COORD. EDUC. E CULTURA",
    "CONTROLE DE RESERVAS",
    "SERVICOS GERAIS",
    "RECEPCAO",
    "GOVERNANCA",
    "GERENCIA OPERACIONAL",
    "LAVANDERIA HOSPEDAGEM",
    "GARAGEM",
    "SAUNA",
    "APARTAMENTOS",
    "COZINHA",
    "BAR",
    "LAVANDERIA INTERNA",
    "PORTARIA",
    "EXCURSOES",
    "ACADEMIA",
    "CURSOS",
    "RESTAURANTE",
    "ADMINISTRATIVO",
    "ALMOXARIFADO",
    "ESTETICA SAUDE E BELEZA",
    "MANUTENCAO",
    "PISCINA",
    "CONSELHO DELIBERATIVO",
    "AUDITORIA INTERNA",
    "ARQUIVO GERAL",
    "GESTAO DE CONTRATOS",
    "OBRAS UR",
    "OBRAS UL",
    "BENFEITORIAS SEDE",
    "BENFEITORIAS UR",
    "GESTAO DE PESSOAS",
    "DIRETORIA ECON. FINANC.",
    "BENFEITORIAS UL",
    "SECRETARIA ADMINISTRATIVA",
    "SECRETARIA ADMINISTRATIVO",
    "SERVICOS GERAIS VB",
    "CONSELHO FISCAL",
    "PRODUCAO E EDICAO AUDIO VISUAL",
    "ESCOLA AFPESP",
    "SECRETARIA DE OBRAS",
    "GESTAO DA QUALIDADE",
    "FAZENDA",
    "OBRAS SEDE",
    "CONTROLES INTERNOS",
    "RECREACAO"
]

# Diretório de downloads
DIRETORIO_DOWNLOADS = os.path.expanduser('~/Downloads')

# Diretório de saída para arquivos renomeados
DIRETORIO_SAIDA = os.path.expanduser('~/Downloads/Centros_Custo_Exportados')

# Função para aguardar o download completar
def aguardar_download(timeout=15):
    """Aguarda até que não haja arquivos .crdownload no diretório de downloads"""
    tempo_inicial = time.time()
    while time.time() - tempo_inicial < timeout:
        crdownload_files = glob.glob(os.path.join(DIRETORIO_DOWNLOADS, '*.crdownload'))
        if not crdownload_files:
            time.sleep(2)  # Aguardar mais um pouco para garantir que o arquivo foi escrito
            return True
        time.sleep(1)
    logging.warning(f"Timeout aguardando download após {timeout} segundos")
    return False

# Função para obter arquivo específico
def obter_arquivo_especifico(nome_arquivo, timeout=15):
    """Aguarda e retorna o caminho de um arquivo específico"""
    tempo_inicial = time.time()
    while time.time() - tempo_inicial < timeout:
        caminho_arquivo = os.path.join(DIRETORIO_DOWNLOADS, nome_arquivo)
        if os.path.exists(caminho_arquivo) and os.path.getsize(caminho_arquivo) > 0:
            return caminho_arquivo
        time.sleep(1)
    return None

# Função para limpar downloads (mantém apenas o arquivo especificado)
def limpar_downloads(manter_arquivo=None):
    """Remove arquivos do diretório de downloads, opcionalmente mantendo um específico"""
    try:
        for arquivo in glob.glob(os.path.join(DIRETORIO_DOWNLOADS, '*')):
            if os.path.isfile(arquivo) and not arquivo.endswith('.crdownload'):
                # Não remover o arquivo que queremos manter
                if manter_arquivo and arquivo == manter_arquivo:
                    continue
                try:
                    os.remove(arquivo)
                    logging.info(f"Arquivo removido: {arquivo}")
                except Exception as e:
                    logging.warning(f"Não foi possível remover {arquivo}: {e}")
    except Exception as e:
        logging.error(f"Erro ao limpar downloads: {e}")

# Função para sanitizar nome de arquivo
def sanitizar_nome(nome):
    """Remove caracteres especiais do nome do arquivo"""
    caracteres_invalidos = r'<>:"/\|?*'
    for char in caracteres_invalidos:
        nome = nome.replace(char, '')
    return nome.strip()

# Função principal
def processar_centros_custo(tipo_extracao):
    """Processa o download de arquivos para cada centro de custo baseado no tipo de extração"""
    
    try:
        # Criar diretório de saída se não existir
        os.makedirs(DIRETORIO_SAIDA, exist_ok=True)
        logging.info(f"Diretório de saída criado/verificado: {DIRETORIO_SAIDA}")
        
        # Ações iniciais (executadas uma única vez)
        logging.info("Iniciando ações iniciais...")
        
        pyautogui.click(x=526, y=301)  # Acompanhamento Financeiro
        time.sleep(1)
        
        # Variáveis para armazenar as coordenadas do campo "Centro de Custo" que mudam no loop
        coord_x_cc = 0
        coord_y_cc = 0

        # Lógica condicional baseada na escolha do usuário
        if tipo_extracao == "Realizado":
            pyautogui.click(x=507, y=443)  # Realizado
            time.sleep(10)
            
            pyautogui.click(x=316, y=349)  # Clica em Anos
            time.sleep(1)
            pyautogui.write("2026") 
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)
            
            pyautogui.click(x=577, y=357)  # Clica em Unidades
            pyautogui.press("del")
            time.sleep(1)
            pyautogui.write("1-SEDE", interval=0.2)
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)

            # Define as coordenadas do centro de custo para o Realizado
            coord_x_cc, coord_y_cc = 384, 352
            pyautogui.click(x=coord_x_cc, y=coord_y_cc)
            pyautogui.press("del")
            time.sleep(1)

        elif tipo_extracao == "Orçado":
            pyautogui.click(x=671, y=430)  # Orçado
            time.sleep(10)
            
            pyautogui.click(x=405, y=350)  # Clica em Anos
            time.sleep(1)
            pyautogui.write("2026") 
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)
            
            pyautogui.click(x=635, y=355)  # Clica em Unidades
            time.sleep(1)
            pyautogui.press("del")
            pyautogui.write("1-SEDE", interval=0.2)
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)
            
            # Define as coordenadas do centro de custo para o Orçado
            coord_x_cc, coord_y_cc = 477, 355
            pyautogui.click(x=coord_x_cc, y=coord_y_cc)
            pyautogui.press("del")
            time.sleep(1)
        
        logging.info("Ações iniciais concluídas. Iniciando loop de centros de custo...")
        
        total_centros = len(CENTROS_CUSTO_SEDE)
        for indice, centro in enumerate(CENTROS_CUSTO_SEDE, 1):
            try:
                logging.info(f"Processando {indice}/{total_centros}: {centro}")
                print(f"[{indice}/{total_centros}] Processando: {centro}")
                
                # Clicar no campo de Centros de Custo usando as coordenadas dinâmicas
                pyautogui.click(x=coord_x_cc, y=coord_y_cc)
                time.sleep(1)
                
                # Escrever o nome do centro
                pyautogui.write(centro, interval=0.2)
                time.sleep(1)
                pyautogui.press("down")
                pyautogui.press("down")
                pyautogui.press("down")
                pyautogui.press("enter")
                time.sleep(2)
                
                # Clicar em Ações
                pyautogui.click(x=1483, y=264)
                time.sleep(1)
                
                # Clicar em Exportação de Planilha
                pyautogui.click(x=1258, y=468)
                time.sleep(3)
                
                if aguardar_download():
                    arquivo_baixado = obter_arquivo_especifico('HspWebGrid.xlsx', timeout=15)
                    
                    if arquivo_baixado and os.path.exists(arquivo_baixado):
                        nome_sanitizado = sanitizar_nome(centro)
                        extensao = Path(arquivo_baixado).suffix
                        novo_nome = f"{nome_sanitizado}{extensao}" # Adicionado o tipo no nome do arquivo
                        novo_caminho = os.path.join(DIRETORIO_SAIDA, novo_nome)
                        
                        contador = 1
                        while os.path.exists(novo_caminho):
                            novo_nome = f"{nome_sanitizado}{contador}{extensao}"
                            novo_caminho = os.path.join(DIRETORIO_SAIDA, novo_nome)
                            contador += 1
                        
                        shutil.move(arquivo_baixado, novo_caminho)
                        logging.info(f"✅ Arquivo renomeado e movido para: {novo_nome}")
                        print(f"   ✅ Arquivo salvo como: {novo_nome}")
                        
                        limpar_downloads()
                    else:
                        logging.error(f"❌ Arquivo HspWebGrid.xlsx não encontrado para {centro}")
                        print(f"   ❌ Erro: Arquivo não encontrado")
                else:
                    logging.error(f"❌ Timeout no download para {centro}")
                    print(f"   ❌ Erro: Timeout no download")
                
                time.sleep(1)
                
            except Exception as e:
                logging.error(f"❌ Erro ao processar {centro}: {str(e)}")
                print(f"   ❌ Erro: {str(e)}")
                continue
        
        logging.info("✅ Automação concluída com sucesso!")
        print("\n" + "=" * 80)
        print("✅ AUTOMAÇÃO CONCLUÍDA COM SUCESSO!")
        print(f"📁 Arquivos salvos em: {DIRETORIO_SAIDA}")
        print("=" * 80)
        
    except Exception as e:
        logging.error(f"❌ Erro geral na automação: {str(e)}")
        print(f"\n❌ Erro geral: {str(e)}")

if __name__ == "__main__":
    print("=" * 80)
    print("AUTOMAÇÃO DE DOWNLOAD - CENTROS DE CUSTO")
    print("=" * 80)
    
    # Caixa de diálogo solicitando a escolha do usuário
    tipo_escolhido = pyautogui.confirm(
        text='Qual base de dados você deseja extrair?', 
        title='Seleção de Extração', 
        buttons=['Realizado', 'Orçado']
    )
    
    # Se o usuário clicar em cancelar ou fechar a janela
    if tipo_escolhido is None:
        print("❌ Operação cancelada pelo usuário.")
        logging.info("Operação cancelada pelo usuário no prompt inicial.")
    else:
        print(f"Opção selecionada: {tipo_escolhido}")
        print(f"Total de centros a processar: {len(CENTROS_CUSTO_SEDE)}")
        print(f"Diretório de saída: {DIRETORIO_SAIDA}")
        print("=" * 80)
        
        # Passa a escolha para a função principal
        processar_centros_custo(tipo_escolhido)

# Acessando o Módulo Realizado - Centros de Custo das UL's e UR's

In [ ]:
# Configuração do logging
logging.basicConfig(
    filename='automacao_centros_custo_ul.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Lista de Unidades de Lazer (UL's)
CENTROS_CUSTO_UL = [
    "104-UR SAO JOSE DO RIO PRETO",
    "105-UL CARAGUATATUBA",
    "106-UL MARESIAS",
    "111-UR CAMPINEIRA",
    "118-UL SAO LOURENCO",
    "126-UR MARILIA",
    "135-UR ARARAQUARA",
    "139-UL POCOS DE CALDAS",
    "140-UL UBATUBA",
    "141-UL IBIRA",
    "406-UR SANTOS",
    "407-UL SAO PEDRO",
    "408-UL ITANHAEM",
    "409-CAPITAL",
    "504-UR SOROCABA",
    "615-UR CAMPINAS",
    "616-UR ARACATUBA",
    "617-UR FRANCA",
    "623-UL CAMPOS DO JORDAO",
    "624-UL SOCORRO",
    "629-UR BOTUCATU",
    "633-UL DOIS CORREGOS",
    "634-UL PERUIBE II",
    "635-UL PERUIBE I",
    "851-UL AMPARO",
    "854-UL LINDOIA",
    "858-UR BAURU",
    "859-UL AREADO",
    "860-UL SERRA NEGRA",
    "862-UR PIRACICABA",
    "866-UR PRESIDENTE PRUDENTE",
    "867-UL AVARE",
    "877-UL MONTE VERDE",
    "884-UL FAZENDA TERMAS DE IBIRA",
    "1048-UR RIBEIRAO PRETO",
    "1453-UR TAUBATE",
    "1454-UR SAO JOSE DOS CAMPOS",
    "1553-UR SAO CARLOS",
    "1671-UR GUARULHOS",
    "1685-UR SAO BERNARDO DO CAMPO",
    "1685-UR BRAGANCA PAULISTA",
    "1734-UL AGUAS DE SAO PEDRO",
    "1746-UL BORACEIA",
    "621-UL GUARUJA"
]

# Diretório de downloads
DIRETORIO_DOWNLOADS = os.path.expanduser('~/Downloads')

# Diretório de saída para arquivos renomeados
DIRETORIO_SAIDA_UL = os.path.expanduser('~/Downloads/Centros_Custo_UL_Exportados')

# Função para aguardar o download completar
def aguardar_download(timeout=15):
    """Aguarda até que não haja arquivos .crdownload no diretório de downloads"""
    tempo_inicial = time.time()
    while time.time() - tempo_inicial < timeout:
        crdownload_files = glob.glob(os.path.join(DIRETORIO_DOWNLOADS, '*.crdownload'))
        if not crdownload_files:
            time.sleep(2)  # Aguardar mais um pouco para garantir que o arquivo foi escrito
            return True
        time.sleep(1)
    logging.warning(f"Timeout aguardando download após {timeout} segundos")
    return False

# Função para obter arquivo específico
def obter_arquivo_especifico(nome_arquivo, timeout=15):
    """Aguarda e retorna o caminho de um arquivo específico"""
    tempo_inicial = time.time()
    while time.time() - tempo_inicial < timeout:
        caminho_arquivo = os.path.join(DIRETORIO_DOWNLOADS, nome_arquivo)
        if os.path.exists(caminho_arquivo) and os.path.getsize(caminho_arquivo) > 0:
            return caminho_arquivo
        time.sleep(1)
    return None

# Função para limpar downloads (mantém apenas o arquivo especificado)
def limpar_downloads(manter_arquivo=None):
    """Remove arquivos do diretório de downloads, opcionalmente mantendo um específico"""
    try:
        for arquivo in glob.glob(os.path.join(DIRETORIO_DOWNLOADS, '*')):
            if os.path.isfile(arquivo) and not arquivo.endswith('.crdownload'):
                # Não remover o arquivo que queremos manter
                if manter_arquivo and arquivo == manter_arquivo:
                    continue
                try:
                    os.remove(arquivo)
                    logging.info(f"Arquivo removido: {arquivo}")
                except Exception as e:
                    logging.warning(f"Não foi possível remover {arquivo}: {e}")
    except Exception as e:
        logging.error(f"Erro ao limpar downloads: {e}")

# Função para sanitizar nome de arquivo
def sanitizar_nome(nome):
    """Remove caracteres especiais do nome do arquivo"""
    caracteres_invalidos = r'<>:"/\|?*'
    for char in caracteres_invalidos:
        nome = nome.replace(char, '')
    return nome.strip()

# Função principal
def processar_centros_custo(tipo_extracao):
    """Processa o download de arquivos para cada centro de custo baseado no tipo de extração"""
    
    try:
        # Criar diretório de saída se não existir
        os.makedirs(DIRETORIO_SAIDA_UL, exist_ok=True)
        logging.info(f"Diretório de saída criado/verificado: {DIRETORIO_SAIDA_UL}")
        
        # Ações iniciais (executadas uma única vez)
        logging.info("Iniciando ações iniciais...")
        
        pyautogui.click(x=526, y=301)  # Acompanhamento Financeiro
        time.sleep(1)
        
        # Variáveis para armazenar as coordenadas do campo "Centro de Custo" que mudam no loop
        coord_x_cc = 0
        coord_y_cc = 0

        # Lógica condicional baseada na escolha do usuário
        if tipo_extracao == "Realizado":
            pyautogui.click(x=507, y=443)  # Realizado
            time.sleep(10)
            
            pyautogui.click(x=311, y=305)  # Clica em Anos
            time.sleep(1)
            pyautogui.write("2026") 
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)

            # Define as coordenadas do centro de custo para o Realizado
            pyautogui.click(x=384, y=352)  # Clica em Centro de Custo
            time.sleep(1)
            pyautogui.press("del")
            pyautogui.write("NA_ENTIDADE", interval=0.2)
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)
            
            coord_x_cc, coord_y_cc = 510, 355 # Clica em Unidades
            pyautogui.click(x=coord_x_cc, y=coord_y_cc)
            pyautogui.press("del")
            time.sleep(2)

        elif tipo_extracao == "Orçado":
            pyautogui.click(x=671, y=430)  # Orçado
            time.sleep(10)
            
            pyautogui.click(x=374, y=355)  # Clica em Anos
            time.sleep(1)
            pyautogui.write("2026") 
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)

            # Define as coordenadas do centro de custo para o Orçado
            pyautogui.click(x=475, y=356)  # Clica em Centro de Custo
            time.sleep(1)
            pyautogui.press("del")
            pyautogui.write("NA_ENTIDADE", interval=0.2)
            time.sleep(1)
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("down")
            pyautogui.press("enter")
            time.sleep(2)
            
            coord_x_cc, coord_y_cc = 619, 352 # Clica em Unidades
            pyautogui.press("del")
            pyautogui.click(x=coord_x_cc, y=coord_y_cc)
            time.sleep(2)
            
        logging.info("Ações iniciais concluídas. Iniciando loop de centros de custo...")
        
        total_centros = len(CENTROS_CUSTO_UL)
        for indice, centro in enumerate(CENTROS_CUSTO_UL, 1):
            try:
                logging.info(f"Processando {indice}/{total_centros}: {centro}")
                print(f"[{indice}/{total_centros}] Processando: {centro}")
                
                # Clicar no campo de Centros de Custo usando as coordenadas dinâmicas
                pyautogui.click(x=coord_x_cc, y=coord_y_cc)
                pyautogui.press("del")
                time.sleep(2)
                
                # Escrever o nome do centro
                pyautogui.write(centro, interval=0.2)
                time.sleep(1)
                pyautogui.press("down")
                pyautogui.press("down")
                pyautogui.press("down")
                pyautogui.press("enter")
                time.sleep(2)
                
                # Clicar em Ações
                pyautogui.click(x=1483, y=264)
                time.sleep(1)
                
                # Clicar em Exportação de Planilha
                pyautogui.click(x=1258, y=468)
                time.sleep(3)
                
                if aguardar_download():
                    arquivo_baixado = obter_arquivo_especifico('HspWebGrid.xlsx', timeout=15)
                    
                    if arquivo_baixado and os.path.exists(arquivo_baixado):
                        nome_sanitizado = sanitizar_nome(centro)
                        extensao = Path(arquivo_baixado).suffix
                        novo_nome = f"{nome_sanitizado}{extensao}" # Adicionado o tipo no nome do arquivo
                        novo_caminho = os.path.join(DIRETORIO_SAIDA_UL, novo_nome)
                        
                        contador = 1
                        while os.path.exists(novo_caminho):
                            novo_nome = f"{nome_sanitizado}{contador}{extensao}"
                            novo_caminho = os.path.join(DIRETORIO_SAIDA_UL, novo_nome)
                            contador += 1
                        
                        shutil.move(arquivo_baixado, novo_caminho)
                        logging.info(f"✅ Arquivo renomeado e movido para: {novo_nome}")
                        print(f"   ✅ Arquivo salvo como: {novo_nome}")
                        
                        limpar_downloads()
                    else:
                        logging.error(f"❌ Arquivo HspWebGrid.xlsx não encontrado para {centro}")
                        print(f"   ❌ Erro: Arquivo não encontrado")
                else:
                    logging.error(f"❌ Timeout no download para {centro}")
                    print(f"   ❌ Erro: Timeout no download")
                
                time.sleep(1)
                
            except Exception as e:
                logging.error(f"❌ Erro ao processar {centro}: {str(e)}")
                print(f"   ❌ Erro: {str(e)}")
                continue
        
        logging.info("✅ Automação concluída com sucesso!")
        print("\n" + "=" * 80)
        print("✅ AUTOMAÇÃO CONCLUÍDA COM SUCESSO!")
        print(f"📁 Arquivos salvos em: {DIRETORIO_SAIDA_UL}")
        print("=" * 80)
        
    except Exception as e:
        logging.error(f"❌ Erro geral na automação: {str(e)}")
        print(f"\n❌ Erro geral: {str(e)}")

if __name__ == "__main__":
    print("=" * 80)
    print("AUTOMAÇÃO DE DOWNLOAD - CENTROS DE CUSTO")
    print("=" * 80)
    
    # Caixa de diálogo solicitando a escolha do usuário
    tipo_escolhido = pyautogui.confirm(
        text='Qual base de dados você deseja extrair?', 
        title='Seleção de Extração', 
        buttons=['Realizado', 'Orçado']
    )
    
    # Se o usuário clicar em cancelar ou fechar a janela
    if tipo_escolhido is None:
        print("❌ Operação cancelada pelo usuário.")
        logging.info("Operação cancelada pelo usuário no prompt inicial.")
    else:
        print(f"Opção selecionada: {tipo_escolhido}")
        print(f"Total de centros a processar: {len(CENTROS_CUSTO_UL)}")
        print(f"Diretório de saída: {DIRETORIO_SAIDA_UL}")
        print("=" * 80)
        
        # Passa a escolha para a função principal
        processar_centros_custo(tipo_escolhido)

time.sleep(5)
pyautogui.hotkey("alt", "f4")

# Mover Arquivos para o Repositório
- Para Realizado: X:\Gestão de Pessoas\Analytics\12 - Gestão de Orçamento\12.1 - Repositório PBI\Realizado\2026
- Para Orçado: X:\Gestão de Pessoas\Analytics\12 - Gestão de Orçamento\12.1 - Repositório PBI\Orçamento\2026

In [ ]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Concluído o remanejamento dos arquivos?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

# Atualização do Power BI - REAL X ORÇADO

In [ ]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestão de Pessoas\Analytics\09 - Power BI\REAL X ORÇADO.pbix"
os.startfile(path_pbi) # Abre o arquivo
time.sleep(30)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(60)
pyautogui.click(x=1293, y=98) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(3)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Resumo de Finalização do Processo

In [ ]:
from datetime import datetime, date

tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')